# Imports

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Load data

In [2]:
appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')
print(appart_df.head())

    id_mutation date_mutation  numero_disposition nature_mutation  \
0  2021-1289505    2021-01-05                   1           Vente   
1  2021-1289506    2021-01-08                   1           Vente   
2  2021-1289527    2021-01-11                   1           Vente   
3  2021-1289529    2021-01-04                   1           Vente   
4  2021-1289537    2021-01-06                   1           Vente   

   valeur_fonciere  adresse_numero adresse_suffixe  \
0          81000.0             2.0               B   
1         165000.0             1.0             NaN   
2         167000.0            11.0             NaN   
3         166000.0            51.0             NaN   
4         180000.0            66.0             NaN   

             adresse_nom_voie adresse_code_voie  code_postal  ...  \
0        RUE DES BASSES LOGES              0020      77210.0  ...   
1        RUE DU MOULIN A VENT              1570      77127.0  ...   
2  PL ANTOINE DE BOUGAINVILLE              0124      

C:\Users\alexi\AppData\Local\Temp\ipykernel_11472\2213143999.py:1: DtypeWarning: Columns (20) have mixed types. Specify dtype option on import or set low_memory=False.
  appart_df = pd.read_csv('datasets_prepd/dvf_appartement.csv')


# Features

In [3]:
features = [
    "longitude",
    "latitude",
    "code_postal",
    "surface_reelle_bati",
    "nombre_pieces_principales",
   # "prix_m2_ref",
    "total_carrez_surface",
    "number_of_lots"
]

target = "valeur_fonciere_actualisee"

X = appart_df[features]
y = appart_df[target]

print("Feature count:", X.shape[1])
print(X.head())
print(y.head())

Feature count: 7
   longitude   latitude  code_postal  surface_reelle_bati  \
0   2.733019  48.416896      77210.0                 57.0   
1   2.553326  48.628821      77127.0                 66.0   
2   2.569526  48.656643      77380.0                 67.0   
3        NaN        NaN      77380.0                 81.0   
4   2.562435  48.665468      77380.0                 60.0   

   nombre_pieces_principales  total_carrez_surface  number_of_lots  
0                        4.0                 59.16             2.0  
1                        3.0                 66.22             1.0  
2                        3.0                 67.40             1.0  
3                        4.0                 81.52             1.0  
4                        3.0                 60.06             1.0  
0     81000.0
1    165000.0
2    167000.0
3    166000.0
4    180000.0
Name: valeur_fonciere_actualisee, dtype: float64


# Make sets : trail and test and validate

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Train size:", X_train.shape)
print("Validation size:", X_val.shape)
print("Test size:", X_test.shape)

Train size: (25592, 7)
Validation size: (5484, 7)
Test size: (5484, 7)


# Ssave

In [5]:
os.makedirs("data_apt", exist_ok=True)

train_df = X_train.copy()
val_df = X_val.copy()
test_df = X_test.copy()

train_df[target] = y_train
val_df[target] = y_val
test_df[target] = y_test

train_df.to_csv("data_apt/appart_train.csv", index=False)
val_df.to_csv("data_apt/appart_val.csv", index=False)
test_df.to_csv("data_apt/appart_test.csv", index=False)

print("Datasets saved.")

Datasets saved.


# Train model

In [6]:
model_appart = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

model_appart.fit(X_train, y_train)

print("Model trained.")

Model trained.


# Evaluation

In [7]:
y_val_pred = model_appart.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

print("Validation RMSE:", rmse)
print("Validation MAE:", mae)
print("Validation R2:", r2)

Validation RMSE: 311061.9968772932
Validation MAE: 74313.89080227495
Validation R2: 0.9710396823014833


In [8]:
df_eval = pd.DataFrame({
    "prix_reel": y_val,
    "prix_predit": y_val_pred
})

df_eval["erreur_pct"] = abs(df_eval["prix_reel"] - df_eval["prix_predit"]) / df_eval["prix_reel"] * 100

print(df_eval.head())

print("Erreur moyenne (%) :", df_eval["erreur_pct"].mean())

       prix_reel    prix_predit  erreur_pct
833     173400.0  267031.257380   53.997265
36270   288750.0  477636.441727   65.415218
7022    750750.0  858149.675000   14.305651
8220    704900.0  791905.473752   12.342953
23680   179300.0  166712.732899    7.020227
Erreur moyenne (%) : 29.57246758883974


In [9]:

y_train_pred = model_appart.predict(X_train)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print("Train RMSE:", rmse_train)
print("Train MAE:", mae_train)
print("Train R2:", r2_train)

Train RMSE: 133974.08227920206
Train MAE: 32622.28712042856
Train R2: 0.9948893852900924


SAVE MODEL

In [10]:
os.makedirs("models", exist_ok=True)

joblib.dump(model_appart, "models/apt_random_forest_model.pkl")

print("Model saved.")

Model saved.


In [11]:
print("Number of features expected:", model_appart.n_features_in_)
print("Feature names:", features)

Number of features expected: 7
Feature names: ['longitude', 'latitude', 'code_postal', 'surface_reelle_bati', 'nombre_pieces_principales', 'total_carrez_surface', 'number_of_lots']
